# 00 -- Clip global meteo to the Netherlands

Spatially clip datasets under a **global** Meteo archive (~50 GB, stays on this PC)
to a Netherlands bounding box and keep data from **2004 onwards**.

Writes the mirrored NL subset into `08_09_baseline_forecasting/data/meteo_nl/` (~47 MB).

**Skip this notebook** if `data/meteo_nl/` is already populated (it is in the hand-in copy).
To re-clip from scratch, set env `CH09_METEO_GLOBAL_DIR` (or `METEO_GLOBAL_DIR_OVERRIDE` in the setup cell) to your local global Meteo folder.


In [1]:
%matplotlib inline

from __future__ import annotations

import io
import re
import shutil
import sys
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

try:
    import rasterio
    from rasterio.io import MemoryFile
    from rasterio.windows import from_bounds
except ImportError:
    import subprocess

    subprocess.check_call([sys.executable, "-m", "pip", "install", "rasterio"])
    import rasterio
    from rasterio.io import MemoryFile
    from rasterio.windows import from_bounds

import os

PACKAGE_ROOT = Path("..").resolve()
if not (PACKAGE_ROOT / "data").exists():
    PACKAGE_ROOT = Path(".").resolve()
    if PACKAGE_ROOT.name == "notebooks":
        PACKAGE_ROOT = PACKAGE_ROOT.parent

sys.path.insert(0, str(PACKAGE_ROOT / "src"))
from soil_tif_utils import SOIL_DATA_FOLDER

NL_DIR = PACKAGE_ROOT / "data" / "meteo_nl"
NL_DIR.mkdir(parents=True, exist_ok=True)

NL_LAT_MIN, NL_LAT_MAX = 50.75, 53.55
NL_LON_MIN, NL_LON_MAX = 3.35, 7.25
NL_BBOX = (NL_LON_MIN, NL_LAT_MIN, NL_LON_MAX, NL_LAT_MAX)
TIMESCALES = ["1", "3", "6", "9", "12", "24", "48"]
TIME_START_YEAR = 2004
TIME_START = pd.Timestamp(f"{TIME_START_YEAR}-01-01")

# Optional: set to a Path(...) if you prefer not to use the env var.
METEO_GLOBAL_DIR_OVERRIDE: Path | None = None

SKIP_CLIP = any(NL_DIR.rglob("*")) and any(p.is_file() for p in NL_DIR.rglob("*"))
METEO_DIR: Path | None = None

if SKIP_CLIP:
    print(f"Package:  {PACKAGE_ROOT}")
    print(f"NL out:   {NL_DIR}")
    print(
        "SKIP_CLIP=True - data/meteo_nl/ already has files. "
        "Skip the remaining cells, or delete/empty meteo_nl to re-clip."
    )
else:
    env_meteo = (os.environ.get("CH09_METEO_GLOBAL_DIR") or "").strip()
    if METEO_GLOBAL_DIR_OVERRIDE is not None:
        METEO_DIR = Path(METEO_GLOBAL_DIR_OVERRIDE).expanduser().resolve()
    elif env_meteo:
        METEO_DIR = Path(env_meteo).expanduser().resolve()
    else:
        raise FileNotFoundError(
            "data/meteo_nl/ is empty and no global Meteo source was set.\n"
            "Either keep the shipped NL clip (skip this notebook), or point at your "
            "~50 GB global archive by:\n"
            "  - setting env CH09_METEO_GLOBAL_DIR, or\n"
            "  - setting METEO_GLOBAL_DIR_OVERRIDE = Path(r'...') in this cell."
        )
    if not METEO_DIR.exists():
        raise FileNotFoundError(f"Source meteo folder not found: {METEO_DIR}")
    print(f"Package:  {PACKAGE_ROOT}")
    print(f"Source:   {METEO_DIR}")
    print(f"NL out:   {NL_DIR}")
    print(f"NL bbox: lat [{NL_LAT_MIN}, {NL_LAT_MAX}], lon [{NL_LON_MIN}, {NL_LON_MAX}]")
    print(f"Time filter: {TIME_START.date()} onwards")


Package:  C:\Users\hoeven\Downloads\Thesis hand-in\Impact-Based-Drought-Forecasting-in-the-Netherlands-Using-LLM-Extracted-News-Data\chapters\09_baseline_forecasting
NL out:   C:\Users\hoeven\Downloads\Thesis hand-in\Impact-Based-Drought-Forecasting-in-the-Netherlands-Using-LLM-Extracted-News-Data\chapters\09_baseline_forecasting\data\meteo_nl
SKIP_CLIP=True - data/meteo_nl/ already has files. Skip the remaining cells, or delete/empty meteo_nl to re-clip.


In [2]:
def human_bytes(n: int) -> str:
    for unit in ("B", "KB", "MB", "GB", "TB"):
        if n < 1024:
            return f"{n:.1f} {unit}"
        n /= 1024
    return f"{n:.1f} PB"


def folder_disk_bytes(folder: Path) -> int:
    if not folder.exists():
        return 0
    return sum(f.stat().st_size for f in folder.rglob("*") if f.is_file())


def ensure_dir(path: Path) -> Path:
    path.mkdir(parents=True, exist_ok=True)
    return path


def print_summary(title: str, lines: list[str]) -> None:
    print(f"\n=== {title} ===")
    for line in lines:
        print(line)


def file_start_year(path: Path) -> int:
    match = re.search(r"_(\d{4})\d{4}_\d{4}", path.name)
    if match:
        return int(match.group(1))
    match = re.search(r"(\d{4})", path.stem)
    return int(match.group(1)) if match else 0


def include_nc_file(path: Path) -> bool:
    return file_start_year(path) >= TIME_START_YEAR


def subset_xarray_nl(ds: xr.Dataset) -> xr.Dataset:
    return ds.sel(
        lat=slice(NL_LAT_MAX, NL_LAT_MIN),
        lon=slice(NL_LON_MIN, NL_LON_MAX),
    )


def netcdf_encoding(ds: xr.Dataset) -> dict:
    encoding = {}
    for var in ds.data_vars:
        if var == "spatial_ref":
            continue
        encoding[var] = {"zlib": True, "complevel": 4}
    return encoding


def subset_netcdf_file(src: Path, dst: Path) -> tuple[int, int]:
    ensure_dir(dst.parent)
    with xr.open_dataset(src) as ds:
        clipped = subset_xarray_nl(ds)
        clipped.to_netcdf(dst, encoding=netcdf_encoding(clipped))
        return int(clipped.sizes["lat"]), int(clipped.sizes["lon"])


def subset_netcdf_folder(src_folder: Path, dst_folder: Path, label: str) -> dict:
    files = sorted(src_folder.glob("*.nc"))
    ensure_dir(dst_folder)
    lat_size = lon_size = 0
    written = skipped = 0
    for src in files:
        if not include_nc_file(src):
            skipped += 1
            print(f"  skipped (before {TIME_START_YEAR}): {src.name}")
            continue
        dst = dst_folder / src.name
        lat_size, lon_size = subset_netcdf_file(src, dst)
        written += 1
        print(f"  [{written}] {src.name} -> {lat_size} x {lon_size}")
    return {"files": written, "skipped": skipped, "lat": lat_size, "lon": lon_size}


def subset_geotiff_bytes(data: bytes, bbox: tuple[float, float, float, float]) -> bytes:
    with MemoryFile(data) as mem:
        with mem.open() as src:
            window = from_bounds(*bbox, transform=src.transform)
            window = window.round_offsets().round_lengths()
            profile = src.profile.copy()
            profile.update(
                height=int(window.height),
                width=int(window.width),
                transform=rasterio.windows.transform(window, src.transform),
            )
            cropped = src.read(1, window=window)
            out_mem = MemoryFile()
            with out_mem.open(**profile) as dst:
                dst.write(cropped, 1)
            return out_mem.read()


def subset_soil_zip(src_zip: Path, dst_zip: Path, bbox: tuple[float, float, float, float]) -> tuple[int, tuple[int, int]]:
    ensure_dir(dst_zip.parent)
    tif_count = 0
    shape = (0, 0)
    with zipfile.ZipFile(src_zip, "r") as zin, zipfile.ZipFile(dst_zip, "w", compression=zipfile.ZIP_DEFLATED) as zout:
        members = sorted(n for n in zin.namelist() if n.lower().endswith(".tif"))
        for member in members:
            cropped = subset_geotiff_bytes(zin.read(member), bbox)
            zout.writestr(member, cropped)
            tif_count += 1
            with MemoryFile(cropped) as mem:
                with mem.open() as src:
                    shape = (src.height, src.width)
    return tif_count, shape


ECA_MISSING = -999999
ECA_PERIOD_NAMES = [
    "annual", "winter_half", "summer_half", "djf", "mam", "jja", "son",
    "jan", "feb", "mar", "apr", "may", "jun", "jul", "aug", "sep", "oct", "nov", "dec",
]


def is_netherlands_station(path: Path) -> bool:
    for line in path.read_text(encoding="utf-8", errors="replace").splitlines()[:12]:
        if "THIS FILE HOLDS DATA FOR STATION" in line:
            return "STATION NETHERLANDS" in line
    return False


def parse_eca_station(path: Path) -> tuple[dict | None, list[dict]]:
    text = path.read_text(encoding="utf-8", errors="replace").splitlines()
    header = {
        "staid": None,
        "station_name": None,
        "latitude": None,
        "longitude": None,
        "elevation": None,
        "source_file": path.name,
    }

    for line in text[:20]:
        if "THIS FILE HOLDS DATA FOR STATION" in line:
            rest = line.split("THIS FILE HOLDS DATA FOR STATION", 1)[1]
            rest = rest.split("(STAID:")[0].strip()
            header["station_name"] = " ".join(rest.split())
        if "STAID:" in line:
            match = re.search(r"STAID:\s*(\d+)", line)
            if match:
                header["staid"] = int(match.group(1))
        if line.startswith("LATITUDE:"):
            d, m, s = re.findall(r"\d+", line)[-3:]
            sign = -1 if "-" in line.split("LATITUDE:")[1][:2] else 1
            header["latitude"] = sign * (int(d) + int(m) / 60 + int(s) / 3600)
        if line.startswith("LONGITUDE:"):
            d, m, s = re.findall(r"\d+", line)[-3:]
            sign = -1 if "-" in line.split("LONGITUDE:")[1][:2] else 1
            header["longitude"] = sign * (int(d) + int(m) / 60 + int(s) / 3600)
        if line.startswith("ELEVATION:"):
            nums = re.findall(r"-?\d+", line.split("ELEVATION:")[1])
            if nums:
                header["elevation"] = int(nums[0])

    if header["staid"] is None or header["latitude"] is None or header["longitude"] is None:
        return None, []

    rows: list[dict] = []
    for line in text:
        line = line.strip()
        if not line or line[0].isalpha():
            continue
        parts = line.split()
        if len(parts) < 3 or not parts[0].isdigit() or not parts[1].isdigit() or len(parts[1]) != 4:
            continue
        year = int(parts[1])
        values = []
        for token in parts[2:]:
            if not token.lstrip("-").isdigit():
                break
            values.append(int(token))
        for period, raw in zip(ECA_PERIOD_NAMES, values):
            cdd_days = np.nan if raw == ECA_MISSING else raw / 100.0
            rows.append({"staid": header["staid"], "year": year, "period": period, "cdd_days": cdd_days})

    return header, rows


def subset_eca_folder(src_folder: Path, dst_folder: Path) -> dict:
    ensure_dir(dst_folder)
    for old_txt in dst_folder.glob("*.txt"):
        old_txt.unlink()

    source_files = sorted(src_folder.glob("*.txt"))
    station_rows: list[dict] = []
    series_rows: list[dict] = []
    copied = 0

    for src in source_files:
        if not is_netherlands_station(src):
            continue
        header, rows = parse_eca_station(src)
        if header is None:
            continue
        shutil.copy2(src, dst_folder / src.name)
        station_rows.append(header)
        series_rows.extend(rows)
        copied += 1
        if copied % 100 == 0:
            print(f"  copied {copied} NL stations...")

    stations_df = pd.DataFrame(station_rows)
    series_df = pd.DataFrame(series_rows)
    if not series_df.empty:
        series_df = series_df[series_df["year"] >= TIME_START_YEAR].reset_index(drop=True)
    stations_path = dst_folder / "eca_nl_stations.csv"
    series_path = dst_folder / "eca_nl_timeseries.csv"
    stations_df.to_csv(stations_path, index=False)
    series_df.to_csv(series_path, index=False)

    return {
        "source_files": len(source_files),
        "copied": copied,
        "timeseries_rows": len(series_df),
        "stations_csv": str(stations_path),
        "timeseries_csv": str(series_path),
    }


def subset_spei_spi_root(src_root: Path, dst_root: Path, prefix: str, label: str) -> dict:
    ensure_dir(dst_root)
    total_files = 0
    skipped = 0
    lat_size = lon_size = 0
    for scale in TIMESCALES:
        src_folder = src_root / scale
        dst_folder = dst_root / scale
        files = sorted(src_folder.glob("*.nc"))
        ensure_dir(dst_folder)
        print(f"  timescale {scale} month ({len(files)} source files)")
        scale_written = 0
        for src in files:
            if not include_nc_file(src):
                skipped += 1
                continue
            dst = dst_folder / src.name
            lat_size, lon_size = subset_netcdf_file(src, dst)
            total_files += 1
            scale_written += 1
        print(f"    written: {scale_written}  skipped (before {TIME_START_YEAR}): {len(files) - scale_written}")
    return {"scales": len(TIMESCALES), "files": total_files, "skipped": skipped, "lat": lat_size, "lon": lon_size}


def plot_nl_map_from_nc(path: Path, var_name: str, title: str) -> None:
    with xr.open_dataset(path) as ds:
        da = ds[var_name]
        if "time" in da.dims:
            da = da.isel(time=-1)
        data = np.squeeze(da.values)
        lon = ds["lon"].values
        lat = ds["lat"].values
    fig, ax = plt.subplots(figsize=(6, 5))
    mesh = ax.pcolormesh(lon, lat, data, shading="auto", cmap="RdYlBu_r")
    ax.set_title(title)
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    fig.colorbar(mesh, ax=ax, shrink=0.8)
    plt.tight_layout()
    plt.show()

## CDI 2012-2026

In [3]:
if SKIP_CLIP:
    print("SKIP_CLIP=True - skipping this cell.")
else:
    cdi_stats = subset_netcdf_folder(
        METEO_DIR / "CDI 2012-2026",
        NL_DIR / "CDI 2012-2026",
        "CDI",
    )
    print_summary("CDI subset complete", [f"Files written: {cdi_stats['files']}", f"Grid: {cdi_stats['lat']} x {cdi_stats['lon']}"])

SKIP_CLIP=True - skipping this cell.


## Soil moisture index anomaly (GeoTIFF zips)

In [4]:
if SKIP_CLIP:
    print("SKIP_CLIP=True - skipping this cell.")
else:
    from soil_tif_utils import SOIL_DATA_FOLDER

    soil_src = METEO_DIR / SOIL_DATA_FOLDER
    soil_dst = NL_DIR / SOIL_DATA_FOLDER
    ensure_dir(soil_dst)

    zip_files = sorted(soil_src.glob("*.zip"))
    soil_tif_total = 0
    soil_shape = (0, 0)
    for i, src_zip in enumerate(zip_files, start=1):
        dst_zip = soil_dst / src_zip.name
        print(f"  [{i}/{len(zip_files)}] {src_zip.name}")
        tif_count, shape = subset_soil_zip(src_zip, dst_zip, NL_BBOX)
        soil_tif_total += tif_count
        soil_shape = shape

    soil_stats = {"zips": len(zip_files), "tifs": soil_tif_total, "shape": soil_shape}
    print_summary(
        "Soil moisture subset complete",
        [f"Zip archives: {soil_stats['zips']}", f"GeoTIFFs: {soil_stats['tifs']}", f"Grid: {soil_shape[0]} x {soil_shape[1]}"],
    )

SKIP_CLIP=True - skipping this cell.


## SPEI 1991-2026 (all timescales)

In [5]:
if SKIP_CLIP:
    print("SKIP_CLIP=True - skipping this cell.")
else:
    spei_stats = subset_spei_spi_root(
        METEO_DIR / "SPEI 1991-2026",
        NL_DIR / "SPEI 1991-2026",
        prefix="spe",
        label="SPEI",
    )
    print_summary(
        "SPEI subset complete",
        [
            f"Timescales: {spei_stats['scales']}",
            f"Files written: {spei_stats['files']}",
            f"Grid: {spei_stats['lat']} x {spei_stats['lon']}",
        ],
    )

SKIP_CLIP=True - skipping this cell.


## SPI 1991-2026 (all timescales)

In [6]:
if SKIP_CLIP:
    print("SKIP_CLIP=True - skipping this cell.")
else:
    spi_stats = subset_spei_spi_root(
        METEO_DIR / "SPI 1991-2026",
        NL_DIR / "SPI 1991-2026",
        prefix="spa",
        label="SPI",
    )
    print_summary(
        "SPI subset complete",
        [
            f"Timescales: {spi_stats['scales']}",
            f"Files written: {spi_stats['files']}",
            f"Grid: {spi_stats['lat']} x {spi_stats['lon']}",
        ],
    )

SKIP_CLIP=True - skipping this cell.


## ECA index CDD (Netherlands stations + CSV export)

In [7]:
if SKIP_CLIP:
    print("SKIP_CLIP=True - skipping this cell.")
else:
    eca_stats = subset_eca_folder(
        METEO_DIR / "ECA_indexCDD",
        NL_DIR / "ECA_indexCDD",
    )
    print_summary(
        "ECA subset complete",
        [
            f"Source station files: {eca_stats['source_files']}",
            f"NL stations copied: {eca_stats['copied']}",
            f"Timeseries rows: {eca_stats['timeseries_rows']}",
            f"CSV: {eca_stats['stations_csv']}",
            f"CSV: {eca_stats['timeseries_csv']}",
        ],
    )

SKIP_CLIP=True - skipping this cell.


## Summary and verification

In [8]:
if SKIP_CLIP:
    n_files = sum(1 for f in NL_DIR.rglob("*") if f.is_file())
    print_summary(
        "meteo_nl already present (clip skipped)",
        [
            f"NL dir: {NL_DIR}",
            f"Files: {n_files}",
            f"Output size: {human_bytes(folder_disk_bytes(NL_DIR))}",
            "Continue with notebooks 01-03 (or frozen results).",
        ],
    )
else:
    summary_rows = [
        {"dataset": "CDI", "files": cdi_stats["files"], "grid_or_notes": f"{cdi_stats['lat']} x {cdi_stats['lon']}, skipped {cdi_stats.get('skipped', 0)}"},
        {"dataset": "Soil moisture", "files": soil_stats["zips"], "grid_or_notes": f"{soil_stats['tifs']} tifs, {soil_stats['shape'][0]} x {soil_stats['shape'][1]}"},
        {"dataset": "SPEI", "files": spei_stats["files"], "grid_or_notes": f"{spei_stats['scales']} scales, skipped {spei_stats['skipped']}, {spei_stats['lat']} x {spei_stats['lon']}"},
        {"dataset": "SPI", "files": spi_stats["files"], "grid_or_notes": f"{spi_stats['scales']} scales, skipped {spi_stats['skipped']}, {spi_stats['lat']} x {spi_stats['lon']}"},
        {"dataset": "ECA CDD", "files": eca_stats["copied"], "grid_or_notes": f"{eca_stats['copied']} NL stations, {eca_stats['timeseries_rows']} ts rows (year>={TIME_START_YEAR})"},
    ]
    print_summary(
        "NL subset summary",
        [
            f"Time filter: {TIME_START.date()} onwards",
            pd.DataFrame(summary_rows).to_string(index=False),
            f"Output size: {human_bytes(folder_disk_bytes(NL_DIR))}",
        ],
    )

    # Verify clipped NetCDF bounds
    sample_cdi = sorted((NL_DIR / "CDI 2012-2026").glob("*.nc"))[0]
    with xr.open_dataset(sample_cdi) as ds:
        print(
            f"CDI NL bounds check: lat [{float(ds.lat.min()):.2f}, {float(ds.lat.max()):.2f}], "
            f"lon [{float(ds.lon.min()):.2f}, {float(ds.lon.max()):.2f}]"
        )

    sample_spei = sorted((NL_DIR / "SPEI 1991-2026" / "12").glob("*.nc"))[0]
    plot_nl_map_from_nc(sample_spei, "spe12", "SPEI-12 NL subset (latest timestep)")


=== meteo_nl already present (clip skipped) ===
NL dir: C:\Users\hoeven\Downloads\Thesis hand-in\Impact-Based-Drought-Forecasting-in-the-Netherlands-Using-LLM-Extracted-News-Data\chapters\09_baseline_forecasting\data\meteo_nl
Files: 1093
Output size: 46.6 MB
Continue with notebooks 01-03 (or frozen results).
